# Phase 4 (Track B / WS5): Prospective Single-Hamiltonian OpenMM Committor Shooting Swarm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thisizkavi-lab/cb1-dynamics/blob/main/notebooks/03_Prospective_OpenMM_Shooting_Colab.ipynb)

### Scientific Purpose: Causal Test of Stage-3 Orthogonal Gating Under a Bitwise-Identical Hamiltonian

Across `519.6 µs` of archival ensembles and all `16` human $\text{CB}_1$ experimental structures (`WS1–WS3`), we discovered that outward cytoplasmic $\text{TM6}$ opening ($R_{\text{TM6}} = d(\text{R214}^{3.50}\text{–D338}^{6.30}\ \text{C}\alpha) > 14\text{ \AA}$) is **not** sufficient for productive intracellular commitment: agonist-only structures (`5XRA`, `5XR8`, `7V3Z`) remain kinetically trapped with inward $\text{F155}^{2.42}/\text{F237}^{4.46}$ rotamers and an unlatched $\text{Y294}^{5.58}\text{–Y397}^{7.53}$ motif (`~5.2 Å`), whereas transducer-coupled complexes flip $\text{F155}^{2.42}/\text{F237}^{4.46}$ outward and lock $\text{Y294}^{5.58}\text{–Y397}^{7.53}$ (`~3.1 Å`).

A skeptical referee can ask: *Could differences across archival trajectories reflect different forcefields, thermostabilizing mutations, ionic buffers (`KCl` vs. `NaCl`), or stitched datasets?*

This notebook executes a **prospective, zero-confounder Transition Path / Committor Shooting Swarm** on Google Colab GPUs (`L4` / `A100`) using **one single wild-type human $\text{CB}_1$ (`UniProt P21554`, `M109–M411`, 303 intact WT residues) construct** solvated in an explicit symmetric `POPC` bilayer with crystallographic cholesterols (`CLR502`, `CLR503`), internal polar-channel `TIP3P-FB` hydration, and `0.15 M NaCl` (`amber14-all.xml` + `amber14/tip3pfb.xml`, `4 fs` HMR `hydrogenMass = 3.0 amu`, `HBonds`, `MonteCarloMembraneBarostat`):

| Prospective Arm | Genotype & OpenMM `System` | Initial $R_{\text{TM6}}$ (`R214–D338`) | Initial Orthogonal Stage-3 Configuration ($t = 0$) | Causal Hypothesis Tested |
| :--- | :--- | :--- | :--- | :--- |
| **`Arm_A_Fake_Open_Trapped`** | **WT `P21554`** (`system_wt_ab.xml.gz`) | **`14.3 ± 0.15 Å`** | **Inward** $\text{F155}^{2.42}$ ($\chi_1 \approx +65^\circ$), **inward** $\text{F237}^{4.46}$ ($\chi_1 \approx -80^\circ$), **unlatched** $\text{Y294–Y397} \approx 5.2\text{ \AA}$ | Steric cleft & unlatched $\text{TM5–TM7}$ trigger rapid cytoplasmic $\text{TM6}$ collapse (`low` $\hat{q}_+$). |
| **`Arm_B_Committed_Open`** | **WT `P21554`** (**BITWISE-IDENTICAL** `system_wt_ab.xml.gz`!) | **`14.3 ± 0.15 Å`** | **Outward** $\text{F155}^{2.42}$ ($\chi_1 \approx -65^\circ$), *trans* $\text{F237}^{4.46}$ ($\chi_1 \approx 180^\circ$), **latched** $\text{Y294–Y397} \approx 3.2\text{ \AA}$ | **Zero Hamiltonian differences vs. Arm A**—any survival/commitment gain is **100% caused by $t=0$ orthogonal switches**! |
| **`Arm_C_Mutant_F237L`** | **`F237L`** (`CB1 -> CB2` swap at `4.46`, `system_mut_f237l_c.xml.gz`) | **`14.3 ± 0.15 Å`** | Identical trapped start as **Arm A** (`inward 155/237`, `unlatched 5.2 Å`), lacking `4.46` phenyl ring | Tests whether evolutionary deletion of the `4.46` aromatic ring (`P21554 -> P34972`) relieves steric trapping! |

---

### Google Colab Compute-Unit Budget (`388.31` Compute Units Available) & Multi-Tab Parallelism

- **Default Campaign:** `15 replicates × 20 ns × 3 arms = 45 unbiased shooting trajectories = 900 ns` total (`10 × 2 ns` checkpointed chunks per trajectory; `100 ps` observable stride).
- **Recommended GPU:** **NVIDIA `L4`** (~`1.8–2.1` Compute Units/hr; best ns-per-Compute-Unit efficiency!).
  - **Interim Balanced Readout (`rep01..rep05` = `15` runs = `300 ns`):** ~`55–65` Compute Units. Because the queue runs **interleaved round-robin (`rep01_A, rep01_B, rep01_C, rep02_A, ...`)**, statistical tests are already balanced and valid at `rep05`!
  - **Full `900 ns` Campaign (`45` runs):** ~`165–190` Compute Units (leaving **~`200` Compute Units spare**).
- **Multi-Tab Acceleration (`2–3x` Faster Wall Time):** Open this notebook in **2 or 3 Colab `L4` tabs simultaneously** mounted to the same Google Drive account. Atomic Drive lock files (`locks/<job_id>.lock.json`) + `2 ns` portable `State` XML checkpointing (`checkpoints/<job_id>.state.xml.gz`) + automatic `>25 min` stale-heartbeat reclamation ensure zero duplicate work and instant crash recovery!

In [ ]:
# Cell 1: Colab GPU + Google Drive + OpenMM CUDA Bootstrap
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/thisizkavi-lab/cb1-dynamics.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_WORK_DIR = Path("/content/drive/MyDrive/cb1-dynamics-phase4/ws5_shooting")
    REPO_ROOT = Path("/content/cb1-dynamics")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=False)

    try:
        import openmm  # noqa: F401
        import mdtraj  # noqa: F401
    except ImportError:
        print("Installing OpenMM (CUDA 12) + scientific stack ...")
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "openmm[cuda12]", "mdtraj", "pyarrow", "pandas",
                "scipy", "scikit-learn", "matplotlib", "seaborn",
            ],
            check=True,
        )
    subprocess.run(["nvidia-smi"], check=False)
else:
    import tempfile
    cwd = Path.cwd().resolve()
    for cand in (cwd, cwd / "cb1-dynamics", cwd.parent):
        if (cand / "src" / "ws5_openmm_shooting").exists():
            REPO_ROOT = cand
            break
    else:
        raise FileNotFoundError(f"Could not locate cb1-dynamics repo root from {cwd}")
    DRIVE_WORK_DIR = Path(tempfile.gettempdir()) / "cb1_ws5_local_preview"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DRIVE_WORK_DIR.mkdir(parents=True, exist_ok=True)

import openmm as mm
print(f"OpenMM {mm.__version__} | Platforms: {[mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]}")
print(f"Repo root      : {REPO_ROOT}")
print(f"Drive queue dir: {DRIVE_WORK_DIR}")

### Step 1. Pre-Flight Hamiltonian Identity & $t = 0$ Seed Verification

Before launching production MD, the cell below verifies programmatically that:
1. **`Arm_A_Fake_Open_Trapped`** and **`Arm_B_Committed_Open`** share a **100% bitwise-identical unrestrained OpenMM `System` (`0` steering/restraint forces)**, identical `WT_P21554` sequence (`303` residues `M109–M411`), and identical atom/solvent/cholesterol topology.
2. All three arms start inside the **matched cytoplasmic $\text{TM6}$ slice ($R_{\text{TM6}} = 14.3 \pm 0.25\text{ \AA}$)** while cleanly separating the Stage-3 orthogonal coordinates (`F155/X237` $\chi_1$, `F155sc–N393CA`, and `Y294–Y397` latch).
3. Runs a `100`-step (`0.4 ps`) `4 fs` HMR warmup benchmark on the active GPU/CPU to measure live `ns/day` throughput.

In [ ]:
# Cell 3: Verify Bitwise-Identical Hamiltonian, Matched t=0 Observables & GPU Throughput
import time
import numpy as np
import pandas as pd
import openmm as mm
import openmm.unit as unit

from src.ws5_openmm_shooting import (
    ARM_SPECS,
    CampaignConfig,
    create_production_simulation,
    extract_ws5_frame_observables,
    load_seed_bundle,
)

SEEDS_DIR = REPO_ROOT / "data" / "openmm_seeds"
bundles = {arm: load_seed_bundle(arm, SEEDS_DIR) for arm in ARM_SPECS}

xml_A = mm.XmlSerializer.serialize(bundles["Arm_A_Fake_Open_Trapped"][1])
xml_B = mm.XmlSerializer.serialize(bundles["Arm_B_Committed_Open"][1])
assert xml_A == xml_B, "CRITICAL: Arm A and Arm B OpenMM Systems are NOT bitwise identical!"

force_types_AB = [type(bundles["Arm_A_Fake_Open_Trapped"][1].getForce(i)).__name__ for i in range(bundles["Arm_A_Fake_Open_Trapped"][1].getNumForces())]
for forbidden in ("CustomBondForce", "CustomTorsionForce", "CustomExternalForce"):
    assert forbidden not in force_types_AB, f"Residual steering restraint {forbidden} found in production System!"

seed_rows = []
for arm_name, (pdb, sys_obj, t_idx) in bundles.items():
    bv = pdb.topology.getPeriodicBoxVectors()
    box_nm = np.array([bv[0][0].value_in_unit(unit.nanometer), bv[1][1].value_in_unit(unit.nanometer), bv[2][2].value_in_unit(unit.nanometer)])
    obs = extract_ws5_frame_observables(
        np.array(pdb.positions.value_in_unit(unit.nanometer)),
        box_nm,
        t_idx,
        time_ns=0.0,
        step=0,
    )
    seed_rows.append({
        "Arm": arm_name,
        "Atoms": pdb.topology.getNumAtoms(),
        "Res237": obs["res237_name"],
        "R214_D338_A": round(obs["R214_D338_CA_A"], 2),
        "F155_chi1_deg": round(obs["F155_chi1_deg"], 1),
        "X237_chi1_deg": round(obs["X237_chi1_deg"], 1),
        "F155sc_N393CA_A": round(obs["F155sc_N393CA_A"], 2),
        "Y294_Y397_OH_A": round(obs["Y294_Y397_OH_A"], 2),
        "Pore_Waters": obs["n_wat_pore"],
        "Wire_Cond": round(obs["wire_conductance_d163_ny"], 2),
        "CHL1_Cleft_A": round(obs["min_CHL1_cleft_A"], 2),
    })

df_seeds = pd.DataFrame(seed_rows)
print("Bitwise Hamiltonian Identity (Arm A == Arm B): VERIFIED | Production Forces:", force_types_AB)
print(df_seeds.to_string(index=False))

# Quick 100-step (0.4 ps) throughput check
cfg_bench = CampaignConfig()
pdb_b, sys_b, _ = bundles["Arm_B_Committed_Open"]
sim_bench, plat_name = create_production_simulation(pdb_b, sys_b, cfg_bench, integrator_seed=101, barostat_seed=202)
bv_bench = pdb_b.topology.getPeriodicBoxVectors()
if bv_bench is not None:
    sim_bench.context.setPeriodicBoxVectors(*bv_bench)
sim_bench.context.setPositions(pdb_b.positions)
sim_bench.context.setVelocitiesToTemperature(310.0 * unit.kelvin, 303)
sim_bench.step(10)  # JIT kernel compile warmup
t0 = time.time()
sim_bench.step(100)
dt_wall = time.time() - t0
ns_day = (0.0004 / max(dt_wall, 1e-6)) * 86400.0
print(f"\nWarmup Benchmark on [{plat_name}]: ~{ns_day:.1f} ns/day (2 ns chunk ~ {2.0 / max(ns_day, 1e-3) * 24 * 60:.1f} min)")
del sim_bench

### Step 2. Crash-Proof, Multi-Tab Google Drive Queue Execution

Configure your tab execution budget below and run Cell 5:
- **`N_REPLICATES_PER_ARM = 15`**, **`TRAJ_DURATION_NS = 20.0`**, **`CHUNK_DURATION_NS = 2.0`** (`45` total jobs = `900 ns`, ordered interleaved round-robin `rep01_A, rep01_B, rep01_C, ...`).
- Set **`MAX_JOBS_THIS_TAB = None`** (or e.g. `15` to stop automatically after `rep01..rep05` = `300 ns`) and **`MAX_WALL_HOURS_THIS_TAB = 22.0`** so Colab Pro sessions exit cleanly after saving the latest `2 ns` checkpoint!
- If a session preempts mid-trajectory, re-running Cell 5 (in this or another tab after the `25 min` stale-lock timeout) resumes seamlessly from the last completed `2 ns` `.state.xml.gz` checkpoint!

In [ ]:
# Cell 5: Launch or Resume Interleaved Prospective Shooting Campaign on Google Drive
from src.ws5_openmm_shooting import CampaignConfig, ShootingCampaignQueue

RUN_CAMPAIGN_NOW = True          # Set False to inspect queue status without running MD
N_REPLICATES_PER_ARM = 15        # 15 reps x 3 arms = 45 unbiased trajectories (900 ns total)
TRAJ_DURATION_NS = 20.0          # 20 ns per shooting trajectory
CHUNK_DURATION_NS = 2.0          # Checkpoint portable State XML + Parquet every 2 ns
MAX_JOBS_THIS_TAB = None         # Optional cap per tab (e.g. 15 for 300 ns interim milestone)
MAX_WALL_HOURS_THIS_TAB = 22.0   # Clean stop before Colab 24h runtime ceiling

cfg = CampaignConfig(
    n_replicates_per_arm=N_REPLICATES_PER_ARM,
    traj_duration_ns=TRAJ_DURATION_NS,
    chunk_duration_ns=CHUNK_DURATION_NS,
)
queue = ShootingCampaignQueue(work_dir=DRIVE_WORK_DIR, seeds_dir=SEEDS_DIR, cfg=cfg, prefer_gpu=True)

status_df = queue.status_summary()
print("=== Current Campaign Queue Summary ===")
print(status_df.groupby(["arm_name", "status"]).size().unstack(fill_value=0))
print(f"Total simulated so far: {status_df['simulated_ns'].sum():.1f} / {status_df['target_ns'].sum():.1f} ns\n")

if RUN_CAMPAIGN_NOW:
    completed_this_session = queue.run_worker_loop(
        max_jobs=MAX_JOBS_THIS_TAB,
        max_wall_hours=MAX_WALL_HOURS_THIS_TAB,
        verbose=True,
    )
    print(f"\nWorker finished {len(completed_this_session)} job(s) in this session.")

### Step 3. Live Statistical Readout, Survival Curves & Mori–Zwanzig Closure

You can run Cell 7 **at any point during or after the campaign** (including right after `rep05` = `300 ns`, or even mid-replicate as soon as any `2 ns` chunks exist in `trajectories/`):
- **Pure $R_{\text{TM6}}$ Stopping-Time Commitment (Zero Target–Feature Circularity):** Absorbing collapse $A = \{R_{\text{TM6}} \le 12.80\text{ \AA}\}$ ($\tau_A$, `q_outcome = 0`) vs. full cavity expansion $B = \{R_{\text{TM6}} \ge 15.30\text{ \AA}\}$ ($\tau_B < \tau_A$, `q_outcome = 1`) or uncollapsed full-horizon active persistence ($\tau_A = \infty$, $t_{\max} \ge 18.0\text{ ns}$, terminal-quarter $\overline{R}_{\text{TM6}} \ge 14.00\text{ \AA}$, `q_outcome = 1`). Neither boundary includes orthogonal microswitches, and all three seeds start strictly inside $(12.80, 15.30)\text{ \AA}$ at $t = 0$.
- **Kaplan–Meier Open-$\text{TM6}$ Survival $\hat{S}(t)$** with Kalbfleisch–Prentice Greenwood 95% confidence bands, **Mantel–Cox log-rank $p$-values**, **Clopper–Pearson 95% CIs** on $\hat{q}_+$, **Fisher's exact test**, and **10,000-shuffle permutation $p$-values** on tail cytoplasmic $R_{\text{TM6}}$.
- **Single-Hamiltonian Mori–Zwanzig Memory Closure (`src.core`, WT `Arm A + Arm B` only, pre-hitting $t \le t_{\text{hit}}$):** Evaluates dynamically lagged velocity/history memory on 1D $R_{\text{TM6}}$ vs. Stage-3 microswitches + explicit water-wire conductance + `CHL1` cleft occupancy strictly inside the undecided transition corridor!

In [ ]:
# Cell 7: Live Prospective Readout Table & 3-Panel Publication Figure
import json
import matplotlib.pyplot as plt
from IPython.display import display

from src.ws5_openmm_shooting import (
    plot_prospective_campaign_readout,
    summarize_prospective_campaign,
)

summary = summarize_prospective_campaign(DRIVE_WORK_DIR / "trajectories")
print(f"Loaded {summary['n_trajectories']} trajectory file(s) ({summary.get('total_simulated_ns', 0.0):.2f} ns total)")

if summary["n_trajectories"] > 0:
    arm_rows = []
    for arm_name, st in summary["arm_stats"].items():
        arm_rows.append({
            "Arm": arm_name,
            "N_runs": st["n_trajectories"],
            "Sim_ns": round(st["total_simulated_ns"], 2),
            "N_Collapsed": st["n_collapsed"],
            "Open_Persist_Rate": f"{st['open_persistence_rate']:.3f} [{st['open_persistence_ci95'][0]:.3f}, {st['open_persistence_ci95'][1]:.3f}]",
            "q_hat_Resolved": f"{st['q_hat_resolved']:.3f} [{st['q_hat_ci95'][0]:.3f}, {st['q_hat_ci95'][1]:.3f}]" if st["n_resolved"] > 0 else "pending",
            "Tail_TM6_Mean_SD_A": f"{st['tail_R214_D338_mean_A']:.2f} ± {st['tail_R214_D338_sd_A']:.2f}",
        })
    display(pd.DataFrame(arm_rows))

    print("\n=== Pairwise Prospective Hypothesis Tests ===")
    print(json.dumps(summary["comparisons"], indent=2))

    if summary["memory_closure"].get("available"):
        print("\n=== Prospective Mori-Zwanzig Memory Closure (src.core) ===")
        print(json.dumps(summary["memory_closure"], indent=2))

    fig_path = DRIVE_WORK_DIR / "ws5_prospective_readout.png"
    fig = plot_prospective_campaign_readout(summary, out_png=fig_path)
    plt.show()
    print(f"Saved figure to: {fig_path}")